# card_seg 真牌微調 — 從部署中的 seg-2 權重出發，用實拍錄影的人工多邊形標註微調

合成資料連兩輪改都沒把真實扇形的「多張牌被併成一個框」壓下來（seg-3 還變差），這次直接用真實畫面：
錄影影格以現有模型預標、人工在 labelme 把併在一起的扇形拆成每張牌，加上沒有牌的影格當背景負樣本。
真實資料很少（數十張），所以：凍結骨幹前段、學習率調低、保留時間上最後幾張當驗證；
另外混入合成資料（同一套合成器）防止遺忘，並用合成驗證集確認沒有退步。
扇形標註：人工 labelme ＋幾何擬合補拆（7 張同尺寸牌繞握點旋轉，擬合整把外框後算每張露出區域）。

**Kaggle 設定**：GPU、Internet On、Add Input：`yun193/cards-seg-real`（images/、labels/、base/ 或 base26/best.pt，依 BASE_DIR）、
`gpiosenka/cards-image-datasetclassification`、`jmexpert/describable-textures-dataset-dtd`

In [ ]:
IMGSZ = 640
EPOCHS = 40
BATCH = 16
LR0 = 0.001
FREEZE = 10                    # 凍結骨幹前 10 層，只調頸部與分割頭，降低對少量資料過擬合
N_VAL = 6                      # 時間上最後幾張有標註的影格當驗證
REPEAT = 10                    # 真牌訓練清單重複次數（約 370 張次，與合成約 1:5）
N_SYN_TRAIN, N_SYN_VAL = 2000, 300
SEED = 0
NB_TAG = "seg26-ft-1"
BASE_DIR = "base26"             # 資料集裡的基底權重資料夾：base = v8（seg-2）、base26 = YOLO26（seg26-1）
OUT_NAME = "card_seg26_ft"
S = IMGSZ
print("NB_TAG=" + NB_TAG)

In [ ]:
import glob, json, math, os, random, shutil
from pathlib import Path
import cv2
import numpy as np


# ---- 內嵌自 synth_cards.py ----
"""撲克牌合成器（整張牌分割與牌值分類兩個 Kaggle notebook 共用，以 `# @include synth_cards.py` 內嵌）。

把 gpiosenka 的單張牌照片以扇形／散放／橫排重疊貼到 DTD 紋理背景上：
- fan：手牌扇形，所有牌繞牌下方同一個樞紐點旋轉（最貼近手拿情境）
- scatter：散放、任意角度；row：橫排重疊（發牌）
之後加整體透視、手指遮擋（膚色橢圓）、模糊、亮度、雜訊、JPEG 壓縮。
render 回傳 (影像, 可見區域多邊形[正規化], 每張牌資訊)；可見面積 < 整張的 1% 的牌不出多邊形（實測常見只露出一條細邊＋牌角的後牌，要標成獨立的牌）。
"""
import glob
import math
import random
from pathlib import Path

import cv2
import numpy as np

S = 640                 # 合成圖邊長
# 合成器版本：3 = 現行（可見 1% 即標、4 成疊牌只露牌角細邊）；2 = seg-2 部署版的設定（可見 4%、無細邊疊牌）
SYNTH_V = globals().get("SYNTH_V", 3)


def find_dir(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits


def load_sources(card_root=None, bg_root=None):
    cards = [p for p in find_dir(f"{card_root or '/kaggle/input'}/**/train/*/*.jpg") if "joker" not in p.lower()]
    bgs = find_dir(f"{bg_root or '/kaggle/input'}/**/dtd/images/*/*.jpg") or \
          [p for p in find_dir(f"{bg_root or '/kaggle/input'}/**/*.jpg") if "texture" in p.lower() or "dtd" in p.lower()]
    return cards, bgs



CARD_AR = 1.4   # 牌高 / 牌寬


def rand_bg(bgs, rng):
    if bgs and rng.random() < 0.85:
        im = cv2.imread(rng.choice(bgs))
        if im is not None:
            h, w = im.shape[:2]
            s = rng.uniform(0.3, 1.0) * min(h, w)
            x, y = rng.uniform(0, w - s), rng.uniform(0, h - s)
            return cv2.resize(im[int(y):int(y + s), int(x):int(x + s)], (S, S))
    c = np.array([rng.randint(0, 255) for _ in range(3)], np.float32)
    noise = np.random.default_rng(rng.randint(0, 1 << 30)).normal(0, 12, (S, S, 3))
    return np.clip(c + noise, 0, 255).astype(np.uint8)


def card_quads(rng):
    n = rng.choice([1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7])
    # stack：幾乎平行、往側邊錯開的疊牌（實測最常被分割合併成一塊的情況）
    mode = rng.choices(["fan", "scatter", "row", "stack"], [0.4, 0.15, 0.15, 0.3])[0]
    quads = []
    if mode == "stack":
        n = max(n, 2)
        h = rng.uniform(0.3, 0.65) * S
        w = h / CARD_AR
        ang = rng.uniform(-40, 40)
        # 每張往側邊露出的寬度：4 成是只露出牌角那一條的細邊（實測最常被併進前牌）
        shift = (rng.uniform(0.06, 0.15) if rng.random() < (0.4 if SYNTH_V >= 3 else 0) else rng.uniform(0.15, 0.5)) * w
        drop = rng.uniform(-0.08, 0.08) * h          # 每張上下微錯
        cx0, cy0 = rng.uniform(0.25, 0.6) * S, rng.uniform(0.3, 0.7) * S
        ca, sa = math.cos(math.radians(ang)), math.sin(math.radians(ang))
        for i in range(n):
            a = math.radians(ang + rng.uniform(-3, 3))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            dx, dy = i * shift, i * drop
            quads.append(local @ R.T + [cx0 + dx * ca - dy * sa, cy0 + dx * sa + dy * ca])
    elif mode == "fan":
        h = rng.uniform(0.28, 0.6) * S
        w = h / CARD_AR
        px, py = rng.uniform(0.25, 0.75) * S, rng.uniform(0.55, 0.95) * S
        base = rng.uniform(-35, 35)
        # 實際手牌常是很密的小角度扇形：一半機率 3~8°，一半 7~18°
        step = (rng.uniform(3, 8) if rng.random() < 0.5 else rng.uniform(7, 18)) * rng.choice([1, -1])
        off = rng.uniform(0.05, 0.25) * h
        a0 = base - step * (n - 1) / 2
        for i in range(n):
            a = math.radians(a0 + step * i)
            local = np.array([[-w / 2, -h + off], [w / 2, -h + off], [w / 2, off], [-w / 2, off]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [px + rng.uniform(-3, 3), py + rng.uniform(-3, 3)])
    elif mode == "row":
        h = rng.uniform(0.2, 0.45) * S
        w = h / CARD_AR
        gap = rng.uniform(0.25, 0.7) * w
        cx0 = rng.uniform(0.15, 0.5) * S
        cy = rng.uniform(0.25, 0.75) * S
        ang = rng.uniform(-20, 20)
        for i in range(n):
            a = math.radians(ang + rng.uniform(-5, 5))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [cx0 + i * gap * math.cos(math.radians(ang)), cy + i * gap * math.sin(math.radians(ang))])
    else:
        for _ in range(n):
            h = rng.uniform(0.15, 0.45) * S
            w = h / CARD_AR
            a = math.radians(rng.uniform(0, 360))
            local = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
            R = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])
            quads.append(local @ R.T + [rng.uniform(0.15, 0.85) * S, rng.uniform(0.15, 0.85) * S])
    # 整體透視（模擬鏡頭角度）
    src = np.float32([[0, 0], [S, 0], [S, S], [0, S]])
    j = S * 0.08
    dst = np.float32([[rng.uniform(-j, j), rng.uniform(-j, j)], [S + rng.uniform(-j, j), rng.uniform(-j, j)],
                      [S + rng.uniform(-j, j), S + rng.uniform(-j, j)], [rng.uniform(-j, j), S + rng.uniform(-j, j)]])
    Hm = cv2.getPerspectiveTransform(src, dst)
    quads = [cv2.perspectiveTransform(q.reshape(-1, 1, 2).astype(np.float32), Hm).reshape(-1, 2) for q in quads]
    return quads, mode


def draw_distractors(img, rng):
    """背景負樣本：門、書櫃、畫框一類的長方形（不標註），教模型「長方形≠牌」。"""
    for _ in range(rng.choice([0, 0, 1, 1, 2, 3])):
        w, h = rng.uniform(0.1, 0.5) * S, rng.uniform(0.15, 0.8) * S
        x, y = rng.uniform(-0.1, 0.9) * S, rng.uniform(-0.1, 0.9) * S
        base = np.array(rng.choice([(200, 225, 235), (90, 140, 190), (230, 230, 225), (60, 90, 140), (180, 200, 210)]),
                        np.float32) + np.random.default_rng(rng.randint(0, 1 << 30)).uniform(-25, 25, 3)
        p1, p2 = (int(x), int(y)), (int(x + w), int(y + h))
        cv2.rectangle(img, p1, p2, tuple(int(v) for v in np.clip(base, 0, 255)), -1)
        kind = rng.random()
        if kind < 0.35:      # 書櫃：橫隔板＋書背
            for k in range(1, rng.randint(2, 5)):
                yy = int(y + h * k / 5)
                cv2.line(img, (p1[0], yy), (p2[0], yy), tuple(int(v * 0.6) for v in base), 3)
                xx = p1[0]
                while xx < p2[0]:
                    bw = rng.randint(4, 14)
                    cv2.rectangle(img, (xx, yy - rng.randint(10, 40)), (xx + bw, yy),
                                  tuple(rng.randint(30, 220) for _ in range(3)), -1)
                    xx += bw + rng.randint(0, 3)
        elif kind < 0.65:    # 畫框：內框＋雜訊內容
            m = int(min(w, h) * rng.uniform(0.08, 0.2))
            inner = img[max(0, p1[1] + m):max(0, p2[1] - m), max(0, p1[0] + m):max(0, p2[0] - m)]
            if inner.size:
                inner[:] = np.random.default_rng(rng.randint(0, 1 << 30)).integers(0, 255, inner.shape, dtype=np.uint8)
                inner[:] = cv2.GaussianBlur(inner, (0, 0), 6)
        else:                # 門：門框線＋門把
            cv2.rectangle(img, p1, p2, tuple(int(v * 0.7) for v in base), 2)
            cv2.circle(img, (int(x + w * 0.85), int(y + h * 0.5)), max(2, int(w * 0.03)), (60, 160, 200), -1)


def render(cards, bgs, rng):
    img = rand_bg(bgs, rng)
    if rng.random() < 0.6:
        draw_distractors(img, rng)
    quads, mode = card_quads(rng)
    masks = []
    full_areas = []
    infos = []   # 每張牌：gpiosenka 類別名（例 "ace of spades"）、整張外框四角、可見遮罩（與 masks 同一物件）
    for q in quads:
        path = rng.choice(cards)
        cim = cv2.imread(path)
        if cim is None:
            continue
        ch, cw = 280, 200
        cim = cv2.resize(cim, (cw, ch))
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [cw, 0], [cw, ch], [0, ch]]), q.astype(np.float32))
        warped = cv2.warpPerspective(cim, M, (S, S), flags=cv2.INTER_LINEAR)
        m = np.zeros((S, S), np.uint8)
        cv2.fillConvexPoly(m, q.astype(np.int32), 255)
        if masks and rng.random() < 0.6:   # 疊在別張牌上：牌緣外側一圈柔和陰影（真牌疊放時白牌之間唯一的邊界線索）
            sh = np.zeros((S, S), np.uint8)
            cv2.fillConvexPoly(sh, q.astype(np.int32), 255)
            sh = cv2.GaussianBlur(cv2.dilate(sh, np.ones((5, 5), np.uint8)), (0, 0), rng.uniform(1.5, 4))
            dark = 1 - rng.uniform(0.15, 0.4) * (sh.astype(np.float32) / 255)
            img[:] = (img * dark[..., None]).astype(np.uint8)
        img[m > 0] = warped[m > 0]
        if rng.random() < 0.4:   # 實際白牌疊白牌幾乎沒有輪廓線，只有一部分畫上細邊
            cv2.polylines(img, [q.astype(np.int32)], True, tuple(int(v) for v in rng.choice([(40, 40, 40), (200, 200, 200), (120, 120, 120)])), 1, cv2.LINE_AA)
        for pm in masks:
            pm[m > 0] = 0
        masks.append(m)
        full_areas.append(cv2.contourArea(q.astype(np.float32)))
        infos.append({"label": Path(path).parent.name, "quad": q, "mask": m})
    # 手指遮擋
    if mode == "fan" and rng.random() < 0.6 or rng.random() < 0.2:
        occ = np.zeros((S, S), np.uint8)
        for _ in range(rng.randint(1, 3)):
            cx, cy = rng.uniform(0.1, 0.9) * S, rng.uniform(0.5, 1.0) * S
            cv2.ellipse(occ, (int(cx), int(cy)), (int(rng.uniform(25, 70)), int(rng.uniform(50, 130))),
                        rng.uniform(0, 180), 0, 360, 255, -1)
        skin = np.array([rng.randint(60, 140), rng.randint(100, 180), rng.randint(150, 230)], np.uint8)
        img[occ > 0] = (img[occ > 0] * 0.15 + skin * 0.85).astype(np.uint8)
        for pm in masks:
            pm[occ > 0] = 0
    # 全域擾動
    alpha, beta = rng.uniform(0.6, 1.35), rng.uniform(-40, 40)
    img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
    if rng.random() < 0.35:
        k = rng.choice([3, 5, 7])
        img = cv2.GaussianBlur(img, (k, k), 0) if rng.random() < 0.6 else cv2.blur(img, (k, 1))
    if rng.random() < 0.4:
        noise = np.random.default_rng(rng.randint(0, 1 << 30)).normal(0, rng.uniform(3, 12), img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    q = rng.randint(35, 95)
    img = cv2.imdecode(cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_COLOR)
    # 標註：可見區域多邊形
    polys = []
    for m, fa in zip(masks, full_areas):
        vis = cv2.countNonZero(m)
        if fa <= 0 or vis < (0.01 if SYNTH_V >= 3 else 0.04) * fa:
            continue
        cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cs, key=cv2.contourArea)
        if cv2.contourArea(c) < max(0.008 * fa, 60):
            continue
        c = cv2.approxPolyDP(c, 0.004 * cv2.arcLength(c, True), True).reshape(-1, 2)
        if len(c) >= 3:
            polys.append(np.clip(c / S, 0, 1))
    return img, polys, infos
# ---- 內嵌結束 synth_cards.py ----
def build_synth(cards, bgs, out_dir: Path):
    for split, n in (("train", N_SYN_TRAIN), ("val", N_SYN_VAL)):
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        rng = random.Random(SEED + (0 if split == "train" else 999))
        for i in range(n):
            img, polys, _ = render(cards, bgs, rng)
            cv2.imwrite(str(out_dir / "images" / split / f"{i:05d}.jpg"), img)
            (out_dir / "labels" / split / f"{i:05d}.txt").write_text(
                "\n".join("0 " + " ".join(f"{v:.5f}" for v in p.reshape(-1)) for p in polys) + ("\n" if polys else ""))


def find_input():
    for p in Path("/kaggle/input").rglob("images"):
        if p.is_dir() and (p.parent / "labels").is_dir():
            return p.parent, next(Path("/kaggle/input").rglob(f"{BASE_DIR}/best.pt"))
    raise FileNotFoundError("找不到 cards-seg-real（images/、labels/、base/ 或 base26/best.pt，依 BASE_DIR）")


def make_lists(root: Path, out: Path, syn: Path):
    """依檔名（時間戳）排序；有牌的影格最後 N_VAL 張當驗證，其餘（含背景）重複 REPEAT 次＋合成訓練集當訓練。"""
    imgs = sorted((root / "images").glob("*.jpg"), key=lambda p: p.name.replace("empty_", ""))
    labeled = [p for p in imgs if (root / "labels" / f"{p.stem}.txt").read_text().strip()]
    val = labeled[-N_VAL:]
    train = [p for p in imgs if p not in val]
    out.mkdir(parents=True, exist_ok=True)
    syn_train = sorted((syn / "images" / "train").glob("*.jpg"))
    (out / "train.txt").write_text("\n".join(str(p) for p in train * REPEAT + syn_train) + "\n")
    (out / "val.txt").write_text("\n".join(str(p) for p in val) + "\n")
    (out / "data.yaml").write_text(f"train: {out / 'train.txt'}\nval: {out / 'val.txt'}\nnames:\n  0: card\n")
    (out / "syn.yaml").write_text(f"path: {syn}\ntrain: images/train\nval: images/val\nnames:\n  0: card\n")
    print("真牌訓練", len(train), "張（×", REPEAT, "）＋合成", len(syn_train), "張；真牌驗證", len(val), "張")
    return out / "data.yaml", out / "syn.yaml"

In [ ]:
if os.path.exists("/kaggle/input"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics>=8.4.0"], check=True)
    from ultralytics import YOLO
    ROOT, BASE = find_input()
    SYN = Path("/tmp/seg_syn")
    if not (SYN / "images" / "val").exists():
        cards, bgs = load_sources()
        assert len(cards) > 1000, "找不到 gpiosenka 牌照片"
        build_synth(cards, bgs, SYN)
    DATA, SYNY = make_lists(ROOT, Path("/tmp/seg_ft"), SYN)
    before = YOLO(str(BASE)).val(data=str(DATA), imgsz=IMGSZ, batch=BATCH, device=0)
    before_syn = YOLO(str(BASE)).val(data=str(SYNY), imgsz=IMGSZ, batch=BATCH, device=0)
    model = YOLO(str(BASE))
    model.train(data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, workers=4, freeze=FREEZE,
                optimizer="AdamW", lr0=LR0, warmup_epochs=2, fliplr=0.0, degrees=10, mosaic=0.5, close_mosaic=5,
                patience=15, seed=SEED, project="/kaggle/working/runs", name="seg_ft", exist_ok=True, plots=True)
    run = Path("/kaggle/working/runs/seg_ft")
    best = YOLO(str(run / "weights" / "best.pt"))
    r = best.val(data=str(DATA), imgsz=IMGSZ, batch=BATCH, device=0)
    rs = best.val(data=str(SYNY), imgsz=IMGSZ, batch=BATCH, device=0)
    metrics = {"base": BASE_DIR, "mask_mAP50": float(r.seg.map50), "mask_mAP50-95": float(r.seg.map),
               "box_mAP50": float(r.box.map50), "real_val_mask_mAP50-95_before": float(before.seg.map),
               "real_val_mask_mAP50_before": float(before.seg.map50),
               "syn_val_mask_mAP50-95": float(rs.seg.map), "syn_val_mask_mAP50-95_before": float(before_syn.seg.map),
               "epochs": EPOCHS, "lr0": LR0, "freeze": FREEZE,
               "nb_tag": NB_TAG}
    print(metrics)
    OUT = Path("/kaggle/working/out") / OUT_NAME; OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy(run / "weights" / "best.pt", OUT / "best.pt")
    for f in ("results.csv", "results.png", "val_batch0_pred.jpg", "val_batch0_labels.jpg"):
        if (run / f).exists():
            shutil.copy(run / f, OUT / f)
    json.dump(metrics, open(OUT / "metrics.json", "w"), indent=2)
    shutil.make_archive("/kaggle/working/cards_seg_ft", "zip", "/kaggle/working/out")